In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

ct_f = ColumnTransformer(
    [
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            make_column_selector(dtype_include=str),
        ),
    ],
    remainder="passthrough",
    sparse_threshold=0,
)

In [ ]:
forest_pipe = Pipeline(
    [
        ("preprocessing", ct_f),
        ("model", RandomForestClassifier(random_state=42)),
    ]
)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_train_b = X_train.copy()
cat_cols = X_train.select_dtypes(include=str).columns
X_train_b[cat_cols] = X_train_b[cat_cols].astype("category")

boosting = HistGradientBoostingClassifier(
    categorical_features="from_dtype", random_state=42
)

forest_cv = cross_val_score(
    forest_pipe, X_train, y_train, cv=cv, scoring="average_precision"
)
boost_cv = cross_val_score(
    boosting, X_train_b, y_train, cv=cv, scoring="average_precision"
)

print(f"RandomForest      CV AP: {np.mean(forest_cv):.3f} +/- {np.std(forest_cv):.3f}")
print(f"HistGradientBoost CV AP: {np.mean(boost_cv):.3f} +/- {np.std(boost_cv):.3f}")

We can see that HGB shows the best performance but it's just a little bit better than log reg. Otherwise RF is the outsider here with a score of 0.37. If I won't be able to boost HGB performance significantly and "business" would like a much cheaper, faster and interpretable model, I will choose Log reg.

I will use `RandomizedSearchCV` in order to understand the upper limit of HGB.

In [ ]:
param_dist = {
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1],
    "max_leaf_nodes": [15, 31, 47, 63],
    "min_samples_leaf": [20, 30, 50, 75, 100],
    "l2_regularization": [0.0, 0.5, 1.0, 2.0, 5.0, 10.0],
    "max_bins": [127, 191, 255],
    "max_depth": [None, 6, 10, 15],
}

hgb = HistGradientBoostingClassifier(
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    class_weight="balanced",
    random_state=42,
)

search = RandomizedSearchCV(
    hgb,
    param_distributions=param_dist,
    n_iter=60,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    random_state=42,
    refit=True,
)

search.fit(X_train_b, y_train)

print("Best AP:", search.best_score_)
print("Best params:", search.best_params_)
best_hgb = search.best_estimator_

- Tuned HGB CV PR-AUC = **~0.42**
- Logistic Regression CV PR-AUC = **0.400 ± 0.025** (PR4)

Tuning left HGB at its ~0.42 ceiling, so it beats LogReg by about 0.02 — real, but inside CV noise. I still choose **Logistic Regression** for the next PRs: at this ceiling the tiny gain isn't worth a heavier, less interpretable model to serve on Lambda. Both numbers are cross-validated on train, the test set is only opened in 07.